In [1]:
# Install dependencies if needed
!pip install pynput ipywidgets pandas

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [pynput]
   ---------------------------------------- 0/2 [pynput]
   ---------------------------------------- 0/2 [pynput]
   -------------------- ------------------- 1/2 [pylsl]
   -------------------- ------------------- 1/2 [pylsl]
   -------------------- ------------------- 1/2 [pylsl]
   ---------------------------------------- 2/2 [pylsl]



# Mouse action log creation
# Does not track scroll wheel alternate mouse buttons aside from left, right click, scroll wheel and mouse movements (outside of scope)
Cyton BCI Board sampling rate = 250 Hz

Mouse log = 100 Hz

rarely need 250 Hz mouse resolution for labeling; 60–125 Hz is plenty to capture direction and clicks.

future steps will aggregate EEG and mouse data into windows (e.g., 250–500 ms) anyway


timestamp = unix Format: floating-point seconds since Jan 1, 1970 (UTC) - provides a simple, precise, and universal time base for synchronizing EEG and mouse data

# ******Confirm long click left and right (hold) as well as  double, triple, etc click tests are recognized, scroll wheel.

In [1]:
import time
import pandas as pd
from pynput import mouse
from pylsl import StreamInfo, StreamOutlet
import ipywidgets as widgets
from IPython.display import display

# First focus on recording events separately
up 

down

right

left

left click (holding, double clicks, etc)

right click (holding, clicking)

In [2]:
left_pressed = False
right_pressed = False
middle_pressed = False  # for wheel click
scroll_delta = 0.0      # for scroll up/down

# ---------------------------
# 1. Set up LSL stream
# ---------------------------


info = StreamInfo(name='MouseStream', type='Mouse', channel_count=4,
                  nominal_srate=100, channel_format='float32', source_id='mouse_001')
outlet = StreamOutlet(info)
print("Mouse LSL stream created.")

# ---------------------------
# 2. Prepare in-memory storage
# ---------------------------
mouse_data = []

# ---------------------------
# 3. Define callback functions
# ---------------------------
def on_click(x, y, button, pressed):
    global left_pressed, right_pressed, middle_pressed
    timestamp = time.time()

    if button.name == 'left':
        left_pressed = pressed
    elif button.name == 'right':
        right_pressed = pressed
    elif button.name == 'middle':
        middle_pressed = pressed

    # Log current state
    mouse_data.append([timestamp, x, y, float(left_pressed), float(right_pressed), float(middle_pressed), 0.0])
    outlet.push_sample([float(x), float(y), float(left_pressed), float(right_pressed), float(middle_pressed), 0.0])

def on_move(x, y):
    global left_pressed, right_pressed, middle_pressed, scroll_delta
    timestamp = time.time()
    mouse_data.append([timestamp, x, y, float(left_pressed), float(right_pressed), float(middle_pressed), float(scroll_delta)])
    outlet.push_sample([float(x), float(y), float(left_pressed), float(right_pressed), float(middle_pressed), float(scroll_delta)])
    scroll_delta = 0.0  # reset after logging to avoid double-counting

def on_scroll(x, y, dx, dy):
    global scroll_delta, left_pressed, right_pressed, middle_pressed
    timestamp = time.time()
    scroll_delta = dy  # positive = up, negative = down

    # Log scroll with current button states
    mouse_data.append([timestamp, x, y, float(left_pressed), float(right_pressed), float(middle_pressed), float(scroll_delta)])
    outlet.push_sample([float(x), float(y), float(left_pressed), float(right_pressed), float(middle_pressed), float(scroll_delta)])

# ---------------------------
# 4. Start listener
# ---------------------------
listener = mouse.Listener(on_move=on_move, on_click=on_click, on_scroll=on_scroll)
listener.start()

print("Mouse logger running. Move the mouse or click buttons.")

# ---------------------------
# 5. Button to stop recording
# ---------------------------
def stop_recording(b):
    listener.stop()
    df = pd.DataFrame(mouse_data, columns=[
    'timestamp', 'x', 'y', 'left_click', 'right_click', 'middle_click', 'scroll_delta'
])
    df.to_csv('mouse_log.csv', index=False)
    print(f"Recording stopped. Saved {len(mouse_data)} samples to 'mouse_log.csv'.")

stop_button = widgets.Button(description="Stop and Save Log")
stop_button.on_click(stop_recording)
display(stop_button)

Mouse LSL stream created.
Mouse logger running. Move the mouse or click buttons.


Button(description='Stop and Save Log', style=ButtonStyle())

# Modify log data to add labels
clss_dir

    -  Higher threshold → less jitter, but very small intentional movements might be ignored.

    -  Lower threshold → more sensitive, but small noise or tremors could be misclassified.

(record some logs and verify only intended classifications exist!)

ex: plan 1 right 1 left 1 down 1 up 2 left clicks 2 right clicks


reg_dy

reg_dy

reg_magnitude

In [3]:
# Load log CSV
df = pd.read_csv("mouse_log.csv")

# Compute regression deltas
# Show directional change along each axis individually
df["reg_dx"] = df["x"].diff().fillna(0)
df["reg_dy"] = df["y"].diff().fillna(0)

# Compute regression magnitude
# Magnitude combines dx and dy into one scalar using Euclidean distance
# (straight-line difference in position between consecutive samples, measured in pixels)
df["reg_magnitude"] = (df["reg_dx"]**2 + df["reg_dy"]**2) ** 0.5

# Classification: direction
# Thresh = the minimum movement in pixels needed to count as a directional action
def direction_label(dx, dy, thresh=2):
    if abs(dx) < thresh and abs(dy) < thresh:
        return "rest"
    if abs(dx) > abs(dy):
        return "right" if dx > 0 else "left"
    else:
        return "down" if dy > 0 else "up"

# Compute direction classification from direction_label of dx, dy
df["clss_dir"] = df.apply(
    lambda r: direction_label(r["reg_dx"], r["reg_dy"]),
    axis=1
)

# Save updated CSV
df.to_csv("mouse_log_labeled.csv", index=False)
print("Saved new log with reg / clss labels")

Saved new log with reg / clss labels
